env & installations

In [1]:
import pandas as pd
import random
import json
from collections import defaultdict
from tqdm.notebook import tqdm

from langchain_groq import ChatGroq
from langchain.chains.summarize import load_summarize_chain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os
import time

load_dotenv()

True

Load and embde responses

In [ ]:
FILE_PATH      = "clustering_output_copy.csv"
COMMENT_COLUMN = "komentar"
CLUSTER_COLUMN = "cluster"

# groq conf
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)
print(f"✓ LLM ready: llama-3.1-8b-instant via Groq")

# cluster filter so it doesn't summarize clusters w/ singel member
MIN_CLUSTER_SIZE         = 3
MAX_COMMENTS_PER_CLUSTER = 40

✓ LLM ready: llama-3.1-8b-instant via Groq


In [3]:
# Load the HAC clustering result
if FILE_PATH.endswith((".xlsx", ".xls")):
    df = pd.read_excel(FILE_PATH)
else:
    df = pd.read_csv(FILE_PATH)

# Quick sanity check
cluster_counts = df[CLUSTER_COLUMN].value_counts()
total_clusters = cluster_counts.shape[0]
small_mask     = cluster_counts < MIN_CLUSTER_SIZE

print(f"Loaded {len(df):,} comments across {total_clusters} clusters")
print(f"Clusters to summarize individually : {(~small_mask).sum()}  (size >= {MIN_CLUSTER_SIZE})")
print(f"Tiny clusters pooled into misc     : {small_mask.sum()}  (size < {MIN_CLUSTER_SIZE})")
print(f"Largest cluster                    : {cluster_counts.max():,} comments")
print()
print("Top 10 clusters by size:")
print(cluster_counts.head(10))

Loaded 1,128 comments across 985 clusters
Clusters to summarize individually : 19  (size >= 3)
Tiny clusters pooled into misc     : 966  (size < 3)
Largest cluster                    : 24 comments

Top 10 clusters by size:
cluster
202    24
638    24
424    18
836     9
664     6
268     6
340     5
588     5
635     5
513     5
Name: count, dtype: int64


## 3. Clustering the Responses

In [4]:
random.seed(42)

# Build clustered_responses — same structure as the original tutorial
grouped = df.groupby(CLUSTER_COLUMN)[COMMENT_COLUMN].apply(list).to_dict()

clustered_responses = defaultdict(list)   # {cluster_id: [text, ...]}
misc_texts = []

for cluster_id, comments in grouped.items():
    comments = [str(c).strip() for c in comments if pd.notna(c) and str(c).strip()]

    if len(comments) < MIN_CLUSTER_SIZE:
        misc_texts.extend(comments)
    else:
        if len(comments) > MAX_COMMENTS_PER_CLUSTER:
            comments = random.sample(comments, MAX_COMMENTS_PER_CLUSTER)
            print(f"  Cluster {cluster_id}: sampled {MAX_COMMENTS_PER_CLUSTER} "
                  f"from {cluster_counts[cluster_id]} comments")
        clustered_responses[cluster_id] = comments

if misc_texts:
    if len(misc_texts) > MAX_COMMENTS_PER_CLUSTER:
        misc_texts = random.sample(misc_texts, MAX_COMMENTS_PER_CLUSTER)
    clustered_responses["misc"] = misc_texts

# Each cluster now contains semantically similar responses — same as the original tutorial.
print(f"\nReady: {len(clustered_responses)} cluster groups")


Ready: 20 cluster groups


## 4. LangChain Map-Reduce Summarization

In [5]:
map_prompt_template = """
Kamu adalah analis teks. Berikut adalah beberapa komentar masyarakat berbahasa Indonesia:

\"{text}\"

Tulis ringkasan singkat (1-2 kalimat) dalam bahasa Indonesia yang menangkap inti komentar-komentar ini.
RINGKASAN:"""

map_prompt = PromptTemplate(template=map_prompt_template, input_variables=["text"])

combine_prompt_template = """
Kamu adalah analis teks. Berikut adalah ringkasan dari beberapa komentar dalam satu kelompok:

\"{text}\"

Gabungkan menjadi satu ringkasan akhir (1-2 kalimat) dalam bahasa Indonesia.
RINGKASAN AKHIR:"""

combine_prompt = PromptTemplate(template=combine_prompt_template, input_variables=["text"])

In [6]:
cluster_summaries = []
cluster_results   = []

for i, (cluster_id, texts) in enumerate(clustered_responses.items()):
    print(f"Summarizing cluster {cluster_id} ({i+1}/{len(clustered_responses)})...")
    documents = [Document(page_content=text) for text in texts]
    try:
        chain = load_summarize_chain(
            llm,
            chain_type="map_reduce",
            map_prompt=map_prompt,
            combine_prompt=combine_prompt,
            verbose=False
        )
        summary = chain.invoke(documents)["output_text"].strip()
        cluster_summaries.append(summary)
        cluster_results.append({
            "cluster_id": cluster_id,
            "n_docs"    : len(documents),
            "ringkasan" : summary,
            "status"    : "ok"
        })
        print(f"  ✓ Done")
    except Exception as e:
        print(f"  ✗ ERROR: {e}")   # now prints the actual error
        cluster_results.append({
            "cluster_id": cluster_id,
            "n_docs"    : len(documents),
            "ringkasan" : f"[ERROR: {e}]",
            "status"    : "error"
        })
    time.sleep(20)

ok = sum(1 for r in cluster_results if r["status"] == "ok")
print(f"\nMap step done: {ok}/{len(cluster_results)} clusters summarized")

Summarizing cluster 64 (1/20)...


d:\Kuliah\PA\pdam-scraper\scrape_instagram\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


  ✓ Done
Summarizing cluster 147 (2/20)...
  ✓ Done
Summarizing cluster 202 (3/20)...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1619 > 1024). Running this sequence through the model will result in indexing errors


  ✓ Done
Summarizing cluster 213 (4/20)...
  ✓ Done
Summarizing cluster 268 (5/20)...
  ✓ Done
Summarizing cluster 340 (6/20)...
  ✓ Done
Summarizing cluster 368 (7/20)...
  ✓ Done
Summarizing cluster 424 (8/20)...
  ✓ Done
Summarizing cluster 430 (9/20)...
  ✓ Done
Summarizing cluster 449 (10/20)...
  ✓ Done
Summarizing cluster 506 (11/20)...
  ✓ Done
Summarizing cluster 513 (12/20)...
  ✓ Done
Summarizing cluster 588 (13/20)...
  ✓ Done
Summarizing cluster 635 (14/20)...
  ✓ Done
Summarizing cluster 638 (15/20)...
  ✓ Done
Summarizing cluster 664 (16/20)...
  ✓ Done
Summarizing cluster 675 (17/20)...
  ✓ Done
Summarizing cluster 708 (18/20)...
  ✓ Done
Summarizing cluster 836 (19/20)...
  ✓ Done
Summarizing cluster misc (20/20)...
  ✓ Done

Map step done: 20/20 clusters summarized


## 5. Reduce Step: Final Global Summary

In [ ]:
reduce_prompt_template = """
Kamu adalah analis yang merangkum keseluruhan komentar masyarakat berbahasa Indonesia.

Berikut adalah ringkasan dari setiap kelompok komentar:

\"{text}\"

Tulis RINGKASAN GLOBAL dalam 3-5 kalimat bahasa Indonesia yang menggambarkan topik-topik utama,
opini umum, dan hal yang paling sering disinggung. Jawab hanya dengan paragraf, tanpa bullet.
RINGKASAN GLOBAL:"""

reduce_prompt = PromptTemplate(template=reduce_prompt_template, input_variables=["text"])

# Wrap each cluster summary as a Document — same as the original tutorial
final_docs = [Document(page_content=s) for s in cluster_summaries]

final_summary_chain = load_summarize_chain(llm, chain_type="stuff", prompt=reduce_prompt)
final_summary = final_summary_chain.invoke(final_docs)["output_text"].strip()

print("Final Summary:\n", final_summary)

Final Summary:
 Masyarakat berbahasa Indonesia mengungkapkan kekecewaan dan kekhawatiran atas kondisi air yang tidak memadai dan tidak stabil, serta gangguan pasokan air yang sering terjadi di berbagai wilayah. Sentimen umum yang paling sering disinggung adalah kekecewaan, kekhawatiran, dan kehilangan harapan terhadap kondisi air yang tidak memuaskan. Topik-topik utama yang paling sering disinggung adalah kekurangan air, gangguan pasokan air, dan kualitas air yang tidak baik, serta kekhawatiran akan dampak kekeringan pada lingkungan dan kehidupan masyarakat.


---
## Theme Labeling (Optional)

Same optional step as the original tutorial: assign a short theme label to each cluster
using the LLM. `llm.invoke(prompt).content` replaces `llm.predict(prompt)` since
LangChain 0.3 deprecated `.predict()`.

In [9]:
theme_prompt = """
Berikut adalah kumpulan komentar masyarakat berbahasa Indonesia dari satu kelompok:
{responses}
Tentukan TEMA UTAMA kelompok ini dalam 3-5 kata bahasa Indonesia.
Jawab hanya dengan tema, tanpa penjelasan.
"""

In [10]:
for i, result in enumerate(cluster_results):
    print(f"Labeling cluster {result['cluster_id']} ({i+1}/{len(cluster_results)})...")
    if result["status"] != "ok":
        result["tema"] = ""
        continue
    joined_text = "\n".join(clustered_responses[result["cluster_id"]])
    theme = llm.invoke(theme_prompt.format(responses=joined_text)).content.strip()
    result["tema"] = theme
    print(f"  Theme: {theme}")

Labeling cluster 64 (1/20)...
  Theme: Terima Kasih
Labeling cluster 147 (2/20)...
  Theme: Pekerjaan Bangunan
Labeling cluster 202 (3/20)...
  Theme: Ketersediaan Air Bersih
Labeling cluster 213 (4/20)...
  Theme: Kerusakan Listrik
Labeling cluster 268 (5/20)...
  Theme: Ketersediaan Air
Labeling cluster 340 (6/20)...
  Theme: Kerusakan Infrastruktur Air
Labeling cluster 368 (7/20)...
  Theme: Kerusakan Air Bersih
Labeling cluster 424 (8/20)...
  Theme: Kebraon Air Mati
Labeling cluster 430 (9/20)...
  Theme: Kebraon
Labeling cluster 449 (10/20)...
  Theme: Kerusakan Listrik
Labeling cluster 506 (11/20)...
  Theme: Komentar Positif
Labeling cluster 513 (12/20)...
  Theme: Selamat Hari Sumpah Pemuda
Labeling cluster 588 (13/20)...
  Theme: Gangguan yang mengganggu
Labeling cluster 635 (14/20)...
  Theme: Kondisi Lingkungan Benowo
Labeling cluster 638 (15/20)...
  Theme: Kerusakan Air PDAM
Labeling cluster 664 (16/20)...
  Theme: Kerusuhan Air Sememi
Labeling cluster 675 (17/20)...
  Th

---
## Export for Dashboard

Two output files:
- `cluster_summaries.csv` — one row per cluster (theme + summary + comment count)
- `global_summary.json` — global paragraph + metadata for the dashboard overview card

In [11]:
original_sizes = df[CLUSTER_COLUMN].value_counts().to_dict()

results_df = pd.DataFrame(cluster_results)
results_df["jumlah_komentar"] = results_df["cluster_id"].map(original_sizes).fillna(results_df["n_docs"])
results_df = results_df.sort_values("jumlah_komentar", ascending=False).reset_index(drop=True)

# Per-cluster CSV
export_df = results_df[["cluster_id", "jumlah_komentar", "tema", "ringkasan"]].copy()
export_df.to_csv("cluster_summaries.csv", index=False, encoding="utf-8-sig")
print("Saved: cluster_summaries.csv")

# Global summary JSON
global_output = {
    "total_komentar"   : int(len(df)),
    "total_klaster"    : int(total_clusters),
    "klaster_dirangkum": int(len(clustered_responses)),
    "ringkasan_global" : final_summary,
    "klaster_utama": [
        {
            "cluster_id"     : str(r["cluster_id"]),
            "jumlah_komentar": int(r["jumlah_komentar"]),
            "tema"           : r.get("tema", ""),
            "ringkasan"      : r["ringkasan"]
        }
        for _, r in results_df[results_df["status"] == "ok"].iterrows()
    ]
}

with open("global_summary.json", "w", encoding="utf-8") as f:
    json.dump(global_output, f, ensure_ascii=False, indent=2)
print("Saved: global_summary.json")

print("\nFinal Summary:\n", final_summary)

Saved: cluster_summaries.csv
Saved: global_summary.json

Final Summary:
 Masyarakat berbahasa Indonesia mengungkapkan kekecewaan dan kekhawatiran atas kondisi air yang tidak memadai dan tidak stabil, serta gangguan pasokan air yang sering terjadi di berbagai wilayah. Sentimen umum yang paling sering disinggung adalah kekecewaan, kekhawatiran, dan kehilangan harapan terhadap kondisi air yang tidak memuaskan. Topik-topik utama yang paling sering disinggung adalah kekurangan air, gangguan pasokan air, dan kualitas air yang tidak baik, serta kekhawatiran akan dampak kekeringan pada lingkungan dan kehidupan masyarakat.
